# 11.6 — Temporal-Difference learning

TD learning updates before the episode ends by trusting a one-step target.

TD learning uses the one-step target $r+\gamma V(s')$ instead of waiting for a complete episode. That makes it sample-efficient, but step size matters because the target uses the learner's own moving estimate. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

SEED = 1106
rng = np.random.default_rng(SEED)
GAMMA = 0.90
ACTIONS = ["up", "right", "down", "left"]
DELTAS = {
    "up": (-1, 0),
    "right": (0, 1),
    "down": (1, 0),
    "left": (0, -1),
}

@dataclass
class GridEnv:
    name: str
    rows: int
    cols: int
    start: int
    states: list
    index_of: dict
    terminal: set
    P: list
    rewards: np.ndarray
    shape_label: str


def discounted_return(rewards, gamma):
    total = 0.0
    power = 1.0
    for reward in rewards:
        total = total + power * reward
        power = power * gamma
    return total


def softmax(logits):
    shifted = np.asarray(logits, dtype=float) - np.max(logits)
    weights = np.exp(shifted)
    return weights / weights.sum()


def move_cell(cell, action, rows, cols, walls):
    dr, dc = DELTAS[action]
    nr = cell[0] + dr
    nc = cell[1] + dc
    candidate = (nr, nc)
    if nr < 0 or nr >= rows:
        return cell
    if nc < 0 or nc >= cols:
        return cell
    if candidate in walls:
        return cell
    return candidate


def build_grid_env(name, rows, cols, start_cell, goal_cells, pit_cells=None, walls=None, step_cost=-0.02, slip=0.0, wind=0.0, bonuses=None):
    pit_cells = set(pit_cells or [])
    walls = set(walls or [])
    bonuses = dict(bonuses or {})
    goal_cells = dict(goal_cells)
    states = []
    for r in range(rows):
        for c in range(cols):
            if (r, c) not in walls:
                states.append((r, c))
    index_of = {cell: i for i, cell in enumerate(states)}
    terminal_cells = set(goal_cells) | pit_cells
    terminal = {index_of[cell] for cell in terminal_cells}
    n_states = len(states)
    rewards = np.zeros(n_states)
    for cell, reward in goal_cells.items():
        rewards[index_of[cell]] = reward
    for cell in pit_cells:
        rewards[index_of[cell]] = -1.0
    for cell, reward in bonuses.items():
        rewards[index_of[cell]] = reward
    P = []
    for state_index, cell in enumerate(states):
        state_rows = []
        for action in ACTIONS:
            if state_index in terminal:
                state_rows.append([(1.0, state_index, 0.0, True)])
                continue
            side_actions = [action, ACTIONS[(ACTIONS.index(action) - 1) % 4], ACTIONS[(ACTIONS.index(action) + 1) % 4]]
            probs = [1.0 - slip, slip / 2.0, slip / 2.0]
            outcomes = {}
            for prob, actual_action in zip(probs, side_actions):
                if prob <= 0.0:
                    continue
                next_cell = move_cell(cell, actual_action, rows, cols, walls)
                windy_cell = move_cell(next_cell, "up", rows, cols, walls)
                wind_options = [(1.0 - wind, next_cell), (wind, windy_cell)]
                for wind_prob, final_cell in wind_options:
                    if wind_prob <= 0.0:
                        continue
                    next_index = index_of[final_cell]
                    done = next_index in terminal
                    reward = step_cost + rewards[next_index]
                    key = (next_index, done, reward)
                    outcomes[key] = outcomes.get(key, 0.0) + prob * wind_prob
            state_rows.append([(prob, ns, rew, done) for (ns, done, rew), prob in outcomes.items()])
        P.append(state_rows)
    shape_label = f"{rows}x{cols}, |S|={n_states}, |A|={len(ACTIONS)}"
    return GridEnv(name, rows, cols, index_of[start_cell], states, index_of, terminal, P, rewards, shape_label)


def two_state_chain():
    return build_grid_env(
        "D1 two-state chain",
        1,
        2,
        (0, 0),
        {(0, 1): 1.0},
        step_cost=0.0,
        slip=0.0,
    )


def build_env_ladder():
    envs = []
    envs.append(two_state_chain())
    envs.append(build_grid_env(
        "D2 slippery 3-state",
        1,
        3,
        (0, 0),
        {(0, 2): 1.0},
        pit_cells={(0, 1)},
        step_cost=-0.01,
        slip=0.20,
    ))
    envs.append(build_grid_env(
        "D3 4x4 gridworld",
        4,
        4,
        (3, 0),
        {(0, 3): 1.0},
        pit_cells={(1, 3)},
        walls={(1, 1), (2, 1)},
        step_cost=-0.03,
        slip=0.05,
    ))
    envs.append(build_grid_env(
        "D4 stochastic windy grid",
        5,
        5,
        (4, 0),
        {(0, 4): 1.2},
        pit_cells={(2, 3), (3, 2)},
        walls={(1, 1), (1, 2), (3, 1)},
        step_cost=-0.04,
        slip=0.15,
        wind=0.20,
    ))
    envs.append(build_grid_env(
        "D5 larger sparse-reward grid",
        8,
        8,
        (7, 0),
        {(0, 7): 2.0},
        pit_cells={(2, 5), (3, 5), (5, 3), (6, 6)},
        walls={(1, 1), (1, 2), (1, 3), (2, 1), (4, 2), (4, 3), (4, 4), (5, 5)},
        step_cost=-0.025,
        slip=0.10,
        wind=0.10,
        bonuses={(7, 1): 0.25},
    ))
    return envs


def q_from_v(env, V, gamma=GAMMA):
    Q = np.zeros((len(env.states), len(ACTIONS)))
    for s in range(len(env.states)):
        for a in range(len(ACTIONS)):
            total = 0.0
            for prob, next_state, reward, done in env.P[s][a]:
                total = total + prob * (reward + gamma * V[next_state] * (not done))
            Q[s, a] = total
    return Q


def policy_evaluation(env, policy, gamma=GAMMA, sweeps=200, tol=1e-10):
    V = np.zeros(len(env.states))
    errors = []
    for sweep in range(sweeps):
        old = V.copy()
        for s in range(len(env.states)):
            if s in env.terminal:
                V[s] = 0.0
                continue
            total = 0.0
            for a in range(len(ACTIONS)):
                for prob, next_state, reward, done in env.P[s][a]:
                    total = total + policy[s, a] * prob * (reward + gamma * old[next_state] * (not done))
            V[s] = total
        errors.append(float(np.max(np.abs(V - old))))
        if errors[-1] < tol:
            break
    return V, np.asarray(errors)


def value_iteration(env, gamma=GAMMA, sweeps=500, tol=1e-10):
    V = np.zeros(len(env.states))
    errors = []
    residuals = []
    for sweep in range(sweeps):
        old = V.copy()
        Q = q_from_v(env, old, gamma)
        for s in range(len(env.states)):
            if s in env.terminal:
                V[s] = 0.0
            else:
                V[s] = np.max(Q[s])
        residual = float(np.max(np.abs(V - old)))
        errors.append(residual)
        residuals.append(residual)
        if residual < tol:
            break
    policy = np.zeros((len(env.states), len(ACTIONS)))
    greedy = np.argmax(q_from_v(env, V, gamma), axis=1)
    for s, action in enumerate(greedy):
        policy[s, action] = 1.0
    return V, policy, np.asarray(errors), np.asarray(residuals)


def policy_iteration(env, gamma=GAMMA, sweeps=80):
    policy = np.ones((len(env.states), len(ACTIONS))) / len(ACTIONS)
    errors = []
    for sweep in range(sweeps):
        V, eval_errors = policy_evaluation(env, policy, gamma=gamma, sweeps=200)
        Q = q_from_v(env, V, gamma)
        greedy = np.argmax(Q, axis=1)
        new_policy = np.zeros_like(policy)
        for s, action in enumerate(greedy):
            new_policy[s, action] = 1.0
        change = float(np.max(np.abs(new_policy - policy)))
        errors.append(change)
        policy = new_policy
        if change == 0.0:
            break
    V, eval_errors = policy_evaluation(env, policy, gamma=gamma, sweeps=300)
    return V, policy, np.asarray(errors)


def run_episode(env, policy, gamma=GAMMA, max_steps=120, epsilon=0.0, start_state=None, rng=None):
    rng = rng or np.random.default_rng(SEED)
    state = env.start if start_state is None else start_state
    trajectory = []
    for step in range(max_steps):
        if state in env.terminal:
            break
        if rng.random() < epsilon:
            action = int(rng.integers(len(ACTIONS)))
        else:
            probs = policy[state] / policy[state].sum()
            action = int(rng.choice(len(ACTIONS), p=probs))
        choices = env.P[state][action]
        probs = np.array([item[0] for item in choices], dtype=float)
        probs = probs / probs.sum()
        choice = int(rng.choice(len(choices), p=probs))
        prob, next_state, reward, done = choices[choice]
        trajectory.append((state, action, reward, next_state, done))
        state = next_state
        if done:
            break
    return trajectory


def episode_return(trajectory, gamma=GAMMA):
    return discounted_return([step[2] for step in trajectory], gamma)


def monte_carlo_value(env, policy, episodes=300, gamma=GAMMA, epsilon=0.10, exploring_starts=False, rng=None):
    rng = rng or np.random.default_rng(SEED)
    V = np.zeros(len(env.states))
    counts = np.zeros(len(env.states))
    errors = []
    V_star, optimal_policy, vi_errors, residuals = value_iteration(env, gamma=gamma)
    for episode in range(episodes):
        start_state = None
        if exploring_starts:
            candidates = [s for s in range(len(env.states)) if s not in env.terminal]
            start_state = int(rng.choice(candidates))
        trajectory = run_episode(env, policy, gamma=gamma, epsilon=epsilon, start_state=start_state, rng=rng)
        G = 0.0
        seen = set()
        for state, action, reward, next_state, done in reversed(trajectory):
            G = reward + gamma * G
            if state in seen:
                continue
            seen.add(state)
            counts[state] = counts[state] + 1.0
            V[state] = V[state] + (G - V[state]) / counts[state]
        errors.append(float(np.max(np.abs(V - V_star))))
    return V, counts, np.asarray(errors)


def td0_value(env, policy, episodes=300, alpha=0.20, gamma=GAMMA, epsilon=0.10, rng=None):
    rng = rng or np.random.default_rng(SEED)
    V = np.zeros(len(env.states))
    errors = []
    V_star, optimal_policy, vi_errors, residuals = value_iteration(env, gamma=gamma)
    for episode in range(episodes):
        trajectory = run_episode(env, policy, gamma=gamma, epsilon=epsilon, rng=rng)
        for state, action, reward, next_state, done in trajectory:
            target = reward + gamma * V[next_state] * (not done)
            V[state] = V[state] + alpha * (target - V[state])
        errors.append(float(np.max(np.abs(V - V_star))))
    return V, np.asarray(errors)


def evaluate_policy_return(env, policy, episodes=80, gamma=GAMMA, rng=None):
    rng = rng or np.random.default_rng(SEED)
    returns = []
    for episode in range(episodes):
        trajectory = run_episode(env, policy, gamma=gamma, rng=rng)
        returns.append(episode_return(trajectory, gamma))
    return float(np.mean(returns))


def immediate_reward_policy(env):
    policy = np.zeros((len(env.states), len(ACTIONS)))
    for s in range(len(env.states)):
        if s in env.terminal:
            policy[s, 0] = 1.0
            continue
        means = []
        for a in range(len(ACTIONS)):
            means.append(sum(prob * reward for prob, next_state, reward, done in env.P[s][a]))
        policy[s, int(np.argmax(means))] = 1.0
    return policy


def uniform_policy(env):
    return np.ones((len(env.states), len(ACTIONS))) / len(ACTIONS)


def value_grid(env, V):
    grid = np.full((env.rows, env.cols), np.nan)
    for idx, cell in enumerate(env.states):
        grid[cell] = V[idx]
    return grid


def policy_grid(env, policy):
    chars = np.full((env.rows, env.cols), " ", dtype=object)
    arrows = np.array(["^", ">", "v", "<"], dtype=object)
    greedy = np.argmax(policy, axis=1)
    for idx, cell in enumerate(env.states):
        chars[cell] = "T" if idx in env.terminal else arrows[greedy[idx]]
    return chars


def plot_value_policy_panels(envs, values, policies, metric_values, metric_name):
    fig, axes = plt.subplots(2, len(envs), figsize=(4 * len(envs), 7))
    for i, env in enumerate(envs):
        ax = axes[0, i]
        image = ax.imshow(value_grid(env, values[i]), cmap="viridis")
        ax.set_title(env.name)
        plt.colorbar(image, ax=ax, fraction=0.046)
        ax = axes[1, i]
        ax.imshow(value_grid(env, values[i]), cmap="viridis")
        arrows = policy_grid(env, policies[i])
        grid = value_grid(env, values[i])
        for r in range(env.rows):
            for c in range(env.cols):
                if not np.isnan(grid[r, c]):
                    ax.text(c, r, arrows[r, c], ha="center", va="center", color="white")
        ax.set_title("greedy policy")
    fig.tight_layout()
    plt.figure(figsize=(7, 3))
    plt.plot(range(1, len(metric_values) + 1), metric_values, marker="o")
    plt.xticks(range(1, len(metric_values) + 1), ["D1", "D2", "D3", "D4", "D5"])
    plt.ylabel(metric_name)
    plt.xlabel("environment rung")
    plt.title(f"{metric_name} across the D1-D5 ladder")
    plt.grid(True, alpha=0.3)
    plt.show()


def print_ladder_preview(envs):
    for env in envs:
        sample = env.states[: min(5, len(env.states))]
        print(f"{env.name}: {env.shape_label}; start={env.states[env.start]}; sample={sample}")


## The concept, built once on D1

We implement TD(0), verify the lesson's one-step target numerically, and then compare stable and unstable bootstrapping on the D5 grid.

Formula: $V(S_t)\leftarrow V(S_t)+\alpha(r_{t+1}+\gamma V(S_{t+1})-V(S_t))$

First assert the exact worked numbers from the lesson: discounted return, one-step TD target, softmax policy weighting, and UCB exploration pressure. These are small enough to verify by hand.

In [ ]:
lesson_return = discounted_return([1.0, 0.0, 2.0], 0.9)
td_target = 1.0 + 0.9 * 0.8
q_new = 0.4 + 0.5 * (td_target - 0.4)
policy_probs = softmax([1.0, 0.0])
expected_reward = policy_probs[0] * 2.0 + policy_probs[1] * 0.0
ucb_index = 0.55 + np.sqrt(2.0 * np.log(20.0) / 5.0)
assert np.isclose(lesson_return, 2.620)
assert np.isclose(td_target, 1.720)
assert np.isclose(q_new, 1.060)
assert np.isclose(np.round(policy_probs[0], 3), 0.731)
assert np.isclose(np.round(policy_probs[1], 3), 0.269)
assert np.isclose(np.round(expected_reward, 3), 1.462)
assert np.isclose(np.round(ucb_index, 3), 1.645)
print(lesson_return, td_target, q_new, policy_probs, expected_reward, ucb_index)

The lesson's TD target is $y=1+0.9\cdot0.8=1.720$; with old value $0.4$ and $\alpha=0.5$, the new value is $1.060$.

In [ ]:
def one_td_update(old_value, reward, next_value, alpha, gamma):
    target = reward + gamma * next_value
    new_value = old_value + alpha * (target - old_value)
    return target, new_value

target, new_value = one_td_update(0.4, 1.0, 0.8, 0.5, 0.9)
assert np.isclose(target, 1.720)
assert np.isclose(new_value, 1.060)
print(target, new_value)

On D1 from zero values, a terminal reward of $1$ with $\alpha=0.5$ moves the start value halfway to $0.5$.

In [ ]:
target, d1_value = one_td_update(0.0, 1.0, 0.0, 0.5, 0.9)
assert np.isclose(target, 1.0)
assert np.isclose(d1_value, 0.5)
print(d1_value)

## The dataset ladder

The family F12 ladder is built inline: D1 two-state chain, D2 slippery three-state, D3 4x4 gridworld, D4 stochastic windy grid, and D5 larger sparse-reward grid.

In [ ]:
envs = build_env_ladder()
print_ladder_preview(envs)

## Run the same method across D1-D5

Collect the plan metric: value-error vs known optimum.

In [ ]:
envs = build_env_ladder()
values = []
policies = []
metrics = []
for env in envs:
    V_star, policy_star, errors, residuals = value_iteration(env)
    V_td, td_errors = td0_value(env, policy_star, episodes=300, alpha=0.20, epsilon=0.08, rng=np.random.default_rng(SEED))
    value_error = float(np.max(np.abs(V_td - V_star)))
    values.append(V_td)
    policies.append(policy_star)
    metrics.append(value_error)
    print(f"{env.name:28s}  {value_error: .3f}")

## Results visualization

The closing figure has value/policy heatmap panels for every environment plus one summary curve over D1-D5.

In [ ]:
plot_value_policy_panels(envs, values, policies, metrics, "value-error vs known optimum")

## Pitfall on the hardest rung

Reproduce the named D5 pitfall, then apply the fix from the lesson.

In [ ]:
env = envs[-1]
V_star, policy_star, errors, residuals = value_iteration(env)
V_high, high_errors = td0_value(env, policy_star, episodes=160, alpha=0.95, epsilon=0.10, rng=np.random.default_rng(SEED))
V_low, low_errors = td0_value(env, policy_star, episodes=160, alpha=0.15, epsilon=0.10, rng=np.random.default_rng(SEED))
print(f"high-alpha final error: {high_errors[-1]:.3f}")
print(f"smaller-alpha final error: {low_errors[-1]:.3f}")
assert low_errors[-1] <= high_errors[-1] or np.mean(low_errors[-20:]) <= np.mean(high_errors[-20:])

## Evaluate it + Practice

- Metric: value-error vs known optimum on D1-D5, compared with a no-skill uniform or immediate-reward baseline.
- Sanity check: D1 must match the hand value and the lesson numbers asserted above.
- Ablation: turn off discounted consequence or coverage and verify the metric worsens.
- Failure signal: residuals stop shrinking, value shapes mismatch, or D5 return drops below the baseline.
- Reproducibility: keep the provided seed and do not download simulators.

Practice prompts:
1. Change $\gamma$ from $0.90$ to $0.70$ and predict which rungs lose the most value before running.

2. Add one wall to D3 and inspect how the optimal policy heatmap reroutes around it.

3. On D5, compare the no-skill uniform policy with the learned or planned policy using the same return metric.